# Day 2: Pandas — New York Taxi Rides

**Goal:** Use Pandas to answer eight business questions about taxi rides.

I will load the data, calculate each answer, and explain what the result means in plain English.

## Load the data

First, I load the CSV directly from the link provided in the task. Then I check the first few rows and the column names so I know what information is available.

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
df = pd.read_csv(url)

print("Rows and columns:", df.shape)
display(df.head())
print("Columns:", df.columns.tolist())

## Prepare the date and time columns

The pickup and dropoff columns contain dates and times. Converting them to datetime values lets me get the pickup hour and day of the week.

In [ ]:
df["pickup"] = pd.to_datetime(df["pickup"], errors="coerce")
df["dropoff"] = pd.to_datetime(df["dropoff"], errors="coerce")

print("Pickup and dropoff columns are ready.")

## Question 1: How many rides are there, and what is the total revenue?

I will count the rows in the dataset and add the values in the `total` column.

In [ ]:
ride_count = len(df)
total_revenue = df["total"].sum()

print(f"Total rides: {ride_count:,}")
print(f"Total revenue: ${total_revenue:,.2f}")
print(
    f"In plain English, the dataset contains {ride_count:,} rides "
    f"with total recorded revenue of ${total_revenue:,.2f}."
)

## Question 2: Which pickup borough has the most rides, and which has the highest average fare?

I will count rides for each pickup borough and calculate the average fare for each borough. Rows without a pickup borough are not included in these borough comparisons.

In [ ]:
borough_data = df.dropna(subset=["pickup_borough"])

rides_by_borough = borough_data["pickup_borough"].value_counts()
average_fare_by_borough = (
    borough_data.groupby("pickup_borough")["fare"].mean().dropna()
)

busiest_borough = rides_by_borough.idxmax()
highest_average_fare_borough = average_fare_by_borough.idxmax()

print("Ride count by pickup borough:")
display(rides_by_borough)

print("Average fare by pickup borough:")
display(average_fare_by_borough.round(2))

print(
    f"In plain English, {busiest_borough} has the most rides, "
    f"while {highest_average_fare_borough} has the highest average fare."
)

## Question 3: What is the average tip as a share of the fare for card payments and for cash?

For each ride with a positive fare, I will divide the tip by the fare. Then I will calculate the average percentage separately for credit-card and cash payments.

In [ ]:
valid_fares = df[df["fare"] > 0].copy()
valid_fares["tip_share_percent"] = (
    valid_fares["tip"] / valid_fares["fare"] * 100
)

payment_rides = valid_fares[
    valid_fares["payment"].isin(["credit card", "cash"])
]

average_tip_share = payment_rides.groupby("payment")["tip_share_percent"].mean()

print("Average tip as a share of fare:")
display(average_tip_share.round(2))

if "credit card" in average_tip_share.index:
    print(f"Credit card: {average_tip_share['credit card']:.2f}%")
if "cash" in average_tip_share.index:
    print(f"Cash: {average_tip_share['cash']:.2f}%")

print(
    "In plain English, these percentages show the average tip relative "
    "to the fare for each payment type, among rides with a positive fare."
)

## Question 4: Which hour of the day has the most pickups?

I will extract the hour from the pickup time and count the rides in each hour. The hour uses the 24-hour clock.

In [ ]:
valid_pickups = df.dropna(subset=["pickup"]).copy()
valid_pickups["pickup_hour"] = valid_pickups["pickup"].dt.hour

rides_by_hour = valid_pickups["pickup_hour"].value_counts().sort_index()
busiest_hour = rides_by_hour.idxmax()
busiest_hour_count = rides_by_hour.max()

print("Rides by pickup hour:")
display(rides_by_hour)

print(
    f"In plain English, {busiest_hour}:00 is the busiest pickup hour, "
    f"with {busiest_hour_count:,} rides."
)

## Question 5: What share of rides end in a different borough from the one they started in?

I will compare pickup and dropoff boroughs. Rides missing either borough are excluded because the two locations cannot be compared reliably.

In [ ]:
known_boroughs = df.dropna(
    subset=["pickup_borough", "dropoff_borough"]
)

different_borough = (
    known_boroughs["pickup_borough"] != known_boroughs["dropoff_borough"]
)

different_count = different_borough.sum()
comparable_count = len(known_boroughs)

if comparable_count > 0:
    different_share = different_count / comparable_count * 100
else:
    different_share = 0

print(f"Rides with both boroughs known: {comparable_count:,}")
print(f"Rides ending in a different borough: {different_count:,}")
print(f"Share: {different_share:.2f}%")
print(
    f"In plain English, {different_share:.2f}% of rides with both "
    "boroughs recorded end in a different borough."
)

## Question 6: What are the top five pickup zones by number of rides?

I will count how often each pickup zone appears and show the five most common zones. Rows with a missing pickup zone are not counted.

In [ ]:
top_five_zones = df["pickup_zone"].dropna().value_counts().head(5)

print("Top five pickup zones:")
display(top_five_zones)

if not top_five_zones.empty:
    print(
        f"In plain English, {top_five_zones.index[0]} is the busiest "
        f"pickup zone in the top five, with {top_five_zones.iloc[0]:,} rides."
    )

## Question 7: How does fare per mile compare for rides over 10 miles and rides under 2 miles?

I will calculate fare divided by distance for each ride. Rides with zero or negative distance are excluded because fare per mile cannot be calculated for them.

In [ ]:
valid_distance = df[df["distance"] > 0].copy()
valid_distance["fare_per_mile"] = (
    valid_distance["fare"] / valid_distance["distance"]
)

long_rides = valid_distance[valid_distance["distance"] > 10]
short_rides = valid_distance[valid_distance["distance"] < 2]

long_average = long_rides["fare_per_mile"].mean()
short_average = short_rides["fare_per_mile"].mean()

print(f"Rides over 10 miles: {len(long_rides):,}")
print(f"Average fare per mile: ${long_average:.2f}")
print(f"Rides under 2 miles: {len(short_rides):,}")
print(f"Average fare per mile: ${short_average:.2f}")

if pd.notna(long_average) and pd.notna(short_average):
    if long_average > short_average:
        comparison = "rides over 10 miles have the higher average fare per mile"
    elif long_average < short_average:
        comparison = "rides under 2 miles have the higher average fare per mile"
    else:
        comparison = "both groups have the same average fare per mile"
    print(f"In plain English, {comparison}.")
else:
    print("In plain English, there is not enough valid data to compare both groups.")

## Question 8: Which day of the week brings in the most revenue?

I will use the pickup date to assign each ride to a day of the week, then add the `total` values for each day.

In [ ]:
valid_days = df.dropna(subset=["pickup"]).copy()
valid_days["day_of_week"] = valid_days["pickup"].dt.day_name()

day_order = [
    "Monday", "Tuesday", "Wednesday", "Thursday",
    "Friday", "Saturday", "Sunday"
]

revenue_by_day = (
    valid_days.groupby("day_of_week")["total"]
    .sum()
    .reindex(day_order)
    .dropna()
    .sort_values(ascending=False)
)

best_day = revenue_by_day.idxmax()
best_day_revenue = revenue_by_day.max()

print("Revenue by pickup day:")
display(revenue_by_day.round(2))

print(
    f"In plain English, {best_day} has the highest recorded revenue "
    f"by pickup day, at ${best_day_revenue:,.2f}."
)

## Final notes

- The results above are calculated from the dataset when the notebook is run.
- Missing boroughs are excluded only from comparisons that require both boroughs.
- Rides with non-positive fares or distances are excluded from calculations that divide by fare or distance.
- For Question 8, revenue is grouped by the pickup day.